# Exercise 2

**Please Note**: We updated the requirements.txt

Please install the new requirements before editing this exercise.

## Import packages

In [1]:
import os

from vll.utils.download import download_mnist
import numpy as np
import matplotlib.pyplot as plt

import skimage
import skimage.io

import torch
import torch.nn.functional as F
from torchvision import transforms

from models.mnist.simple_cnn import Net

## Task 1
(2 points)

In this task, you will learn some basic tensor operations using the PyTorch library.

Reference for torch: https://pytorch.org/docs/stable/torch.html

In [2]:
# Create a numpy array that looks like this: [0, 1, 2, ..., 19]
arr = np.arange(0,20,1)

# Convert the numpy array to a torch tensor
tensor = torch.from_numpy(arr)
print(tensor)


tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])


In [3]:
# Create a numpy array that looks like this: [0, 1, 2, ..., 19]
arr = np.arange(0,20,1)

# Convert the numpy array to a torch tensor
tensor = torch.from_numpy(arr).float()
print(tensor)

# Create a tensor that contains random numbers.
# It should have the same size like the numpy array.
# Multiply it with the previous tensor.
rand_tensor = torch.rand_like(tensor).float()
tensor = rand_tensor * tensor
print(tensor)

# Create a tensor that contains only 1s.
# It should have the same size like the numpy array.
# Substract it from the previous tensor.
tensor = torch.ones_like(tensor)
print(tensor)

# Get the 5th element using a index.
element = tensor[5]
print(element)

# Create a tensor that contains only 0s.
# It should have the same size like the numpy array.
# Multiply it with the previous tensor without any assignment (in place).


tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15., 16., 17., 18., 19.])
tensor([ 0.0000,  0.1424,  0.0952,  1.2230,  3.5259,  0.3420,  2.7743,  1.6864,
         6.0201,  5.4766,  2.4292,  5.5534,  6.7046,  8.3108,  6.3983,  6.8090,
        11.0972,  8.5351, 15.6502,  1.1783])
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1.])
tensor(1.)


In [4]:
# Load the image from the last exercise as RGB image.
image = skimage.io.imread('./data/pepo.jpg')


# Convert the image to a tensor
image = torch.from_numpy(image)

# Print its shape
original_shape = image.shape
print(image.shape)

# Flatten the image
image = image.reshape(-1)
print(len(image))

# Add another dimension resulting in a 1x78642 tensor
image = image.unsqueeze(0)
print(image.shape)

# Revert the last action
image = image.squeeze(0)
print(image.shape)

# Reshape the tensor, so that it has the original 2D dimensions
image = image.reshape(original_shape)
print(image.shape)

# Calculate the sum, mean and max of the tensor
'''
print(torch.sum(image))
print(torch.mean(image))
print(torch.max(image))
'''

# --> not exectuted the sum mean and max operation since this takes to long....


torch.Size([512, 512, 3])
786432
torch.Size([1, 786432])
torch.Size([786432])
torch.Size([512, 512, 3])


'\nprint(torch.sum(image))\nprint(torch.mean(image))\nprint(torch.max(image))\n'

## Task 2
(2 points)

Use Autograd to perform operations on a tensor and output then gradients.

In [13]:
# Create a random 2x2 tensor which requires gradients
x = torch.rand(2, 2, requires_grad=True)
print(x)

# Create another tensor by adding 2.0
y = x + 2.0
print(y)

# Create a third tensor z = y^2
z = y**2
print(z)

# Compute out as the mean of values in z
out = torch.mean(z)
print(out)

# Perform back propagation on out
out.backward()

# Print the gradients dout/dx
print(x.grad)

# Create a copy of y without gradients
y2 = y.detach()
print(y2.requires_grad)

# Perform the mean operation on z with gradients globally disabled
with torch.no_grad():
    out2 = torch.mean(z)

print(out2.requires_grad)

tensor([[0.5020, 0.2369],
        [0.9227, 0.4296]], requires_grad=True)
tensor([[2.5020, 2.2369],
        [2.9227, 2.4296]], grad_fn=<AddBackward0>)
tensor([[6.2602, 5.0039],
        [8.5422, 5.9029]], grad_fn=<PowBackward0>)
tensor(6.4273, grad_fn=<MeanBackward0>)
tensor([[1.2510, 1.1185],
        [1.4614, 1.2148]])
False
False


## Task 3
(3 points)

Implement a Dataset class for MNIST.

In [ ]:
# We first download the MNIST dataset
download_mnist()

In [ ]:
class MNIST:
    """
    Dataset class for MNIST
    """

    def __init__(self, root, transform=None):
        """
        root -- path to either "training" or "testing"
        
        transform -- transform (from torchvision.transforms)
                     to be applied to the data
        """
        # save transforms
        self.transform = transform
        
        # TODO: create a list of all subdirectories (named like the classes) 
        #       within the dataset root
        self.classes = sorted([
            class_name for class_name in os.listdir(root)
            if os.path.isdir(os.path.join(root, class_name))
        ])
        
        # TODO: create a list of paths to all images
        #       with the ground truth label
        self.images = []
        for class_name in self.classes:
            label = int(class_name)
            class_dir = os.path.join(root, class_name)
            for filename in sorted(os.listdir(class_dir)):
                image_path = os.path.join(class_dir, filename)
                if os.path.isfile(image_path):
                    self.images.append((image_path, label))
        
    
    def __len__(self):
        """
        Returns the lenght of the dataset (number of images)
        """
        # TODO: return the length (number of images) of the dataset
        return len(self.images)

    def __getitem__(self, index):
        """
        Loads and returns one image as floating point numpy array
        
        index -- image index in [0, self.__len__() - 1]
        """
        # TODO: load the ith image as an numpy array (dtype=float32)
        image_path, label = self.images[index]
        image = skimage.io.imread(image_path).astype(np.float32) / 255.0
        
        # TODO: apply transforms to the image (if there are any)
        if self.transform is not None:
            image = self.transform(image)
        
        # TODO: return a tuple (transformed image, ground truth)
        return image, label

## Task 4
(3 points)

You can now load a pretrained neural network model we provide.
Your last task is to run the model on the MNIST test dataset, plot some example images with the predicted labels and compute the prediction accuracy.

In [ ]:
def validate(model, data_loader):
    # TODO: Create a 10x10 grid of subplots
   
    
    model.eval()
    correct = 0 # count for correct predictions
    
    with torch.no_grad():
        for i, item in enumerate(data_loader):
            # TODO: unpack item into image and ground truth
            #       and run network on them
            
            
            # TODO: get class with highest probability
            
            
            # TODO: check if prediction is correct
            #       and add it to correct count
            
            
            # plot the first 100 images
            if i < 100:
                # TODO: compute position of ith image in the grid
                
                
                # TODO: convert image tensor to numpy array
                #       and normalize to [0, 1]
                
                
                # TODO: make wrongly predicted images red
                
                
                # TODO: disable axis and show image
                
                
                # TODO: show the predicted class next to each image
                
            
            elif i == 100:
                plt.show()
    
    # TODO: compute and print the prediction accuracy in percent
    

# create a DataLoader using the implemented MNIST dataset class
data_loader = torch.utils.data.DataLoader(
    MNIST('data/mnist/testing',
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])),
    batch_size=1, shuffle=True)

# create the neural network
model = Net()

# load the statedict from 'models/mnist/simple_cnn.pt'
model.load_state_dict(torch.load('models/mnist/simple_cnn.pt'))

# validate the model
validate(model, data_loader)